In [0]:
import os

from dotenv import load_dotenv
from loguru import logger
from pyspark.sql import SparkSession

from arxiv_curator.config import get_env, load_config
from arxiv_curator.data_processor import DataProcessor
from arxiv_curator.utils import is_databricks
# from arxiv_curator.vector_search import VectorSearchManager

In [ ]:
# import json
# import os
# import re
# import time

# import arxiv
# from loguru import logger
# from pyspark.sql import SparkSession
# from pyspark.sql import types as T
# from pyspark.sql.functions import (
#     col,
#     concat_ws,
#     current_timestamp,
#     explode,
#     udf,
# )
# from pyspark.sql.types import ArrayType, StringType, StructField, StructType

# from arxiv_curator.config import ProjectConfig

## Initialize Configuration & Spark Session

In [0]:
# Init Spark session
if not is_databricks():
    load_dotenv()
    profile = os.environ.get("PROFILE", "DEV")
    logger.info(f"Profile : {profile} (Execution outside Databricks)")

spark = SparkSession.builder.getOrCreate()
logger.info(f"Create Spark Session with the cluster: {spark.conf.get('spark.databricks.clusterUsageTags.clusterId')}")

In [0]:
# Load config
env = get_env(spark)
logger.info(f"Loading configuration (env={env})")
cfg = load_config("project_config.yml", env)
logger.info(f"Configuration loaded: catalog={cfg.catalog}schema={cfg.schema} volume={cfg.volume}")

## Test CLass

In [0]:
processor = DataProcessor(spark=spark, config=cfg)

In [0]:
# Step 1: Download papers and store metadata
records = processor.download_and_store_papers()

In [0]:
# Step 2: Parse PDFs with ai_parse_document
processor.parse_pdfs_with_ai()
logger.info("Parsed documents.")

In [0]:
# Step 3: Process chunks
# processor.end = 2026032218
processor.process_chunks()
logger.info("Processing complete!")

In [0]:
logger.info(f"Processing parsed documents from {processor.papers_table}")
df_papers_table = spark.table(processor.papers_table)
display(df_papers_table)

In [0]:
logger.info(f"Processing parsed documents from {processor.parsed_table}")
df_parsed_table = spark.table(processor.parsed_table)
display(df_parsed_table)
# df_parsed_table_filtered = df_parsed_table.filter(~col("path").rlike("2603\\.18427v1|2603\\.18493v1"))
# display(df_parsed_table_filtered)
# df_parsed_table_filtered.write.format("delta").mode("overwrite").saveAsTable(f"{processor.parsed_table}")

In [0]:
logger.info(f"Processing parsed documents from {processor.chunks_table}")
df_chunks_table = spark.table(processor.chunks_table)
display(df_chunks_table)
print(df_chunks_table.select("arxiv_id").distinct().count())